# rotation-matrix-3d-y-axis — worked example 2: Composing two Y-rotations adds their angles

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rotation-matrix-3d-y-axis`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import math
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

Rotations about the same axis form a one-parameter group: R_y(α) · R_y(β) = R_y(α + β). This means you can compose many small rotations into one large one just by summing the angles, without repeated matrix multiplication. Numerically, the product of two rotation matrices should equal the rotation matrix built from the summed angle, up to floating-point precision.

## Worked solution

**Step 1 — Build a helper to construct R_y.** We'll reuse it for three different angles: α, β, and α+β.

**Step 2 — Compose by matrix multiplication.** `R_y(α) @ R_y(β)` is standard matrix multiplication in PyTorch with the `@` operator.

**Step 3 — Build R_y(α+β) independently.** This is our ground-truth comparison. If the group property holds, both matrices should be elementwise equal within floating-point noise.

**Step 4 — Compute the max absolute difference.** `(composed - direct).abs().max()` should be a tiny number (< 1e-6) for any angles. We print this to confirm.

**Why this works.** Euler's formula for rotations in the X-Z plane shows that the cos/sin addition identities are exactly what matrix multiplication produces for single-axis rotations.

In [ ]:
import torch as t
import math

t.manual_seed(7)

def make_ry(deg: float) -> t.Tensor:
    theta = math.radians(deg)
    c, s = math.cos(theta), math.sin(theta)
    return t.tensor([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]], dtype=t.float32)

alpha_deg, beta_deg = 37.0, 53.0
Ra = make_ry(alpha_deg)
Rb = make_ry(beta_deg)
composed = Ra @ Rb
direct = make_ry(alpha_deg + beta_deg)
err = (composed - direct).abs().max().item()
print(f'R({alpha_deg}°) @ R({beta_deg}°) == R({alpha_deg + beta_deg}°)?')
print(f'  max |diff| = {err:.2e}  (should be near machine epsilon)')
print('Composed:\n', composed.round(decimals=4))
print('Direct  :\n', direct.round(decimals=4))